# TIGER: Generative Retrieval with Constrained Beam Search

**Part 2d of the Sequential Recommenders series**

TIGER flattens user history into a single token stream and does **next-token prediction** — exactly like a language model.
At inference, it generates the next item's Semantic ID using **constrained beam search** over a trie of all valid Semantic IDs.

Key differences from Semantic SASRec:
- Input is a **flat token stream** (all levels concatenated), not item-level sequences
- Training objective is **cross-entropy on every token position**
- Inference uses **beam search** constrained by a trie, not candidate scoring

**Prerequisites:** Run the RQVAE notebook (checkpoints 1-2), then the SASRec notebook (checkpoint 3).


---
## 1. Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
import gc
import ast
import gzip
import json
import time
import pickle
import subprocess
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import List, Optional, Tuple, Dict, NamedTuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from tqdm.notebook import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"

%matplotlib inline
sns.set_style('whitegrid')

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")
if device == 'cuda':
    torch.set_float32_matmul_precision('high')
    print(f"GPU: {torch.cuda.get_device_name()}")

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: cuda
GPU: NVIDIA A10G


---
## Load Checkpoints

Load pre-computed data from the RQVAE and SASRec notebooks.


In [2]:
# ============================================================
# Load Checkpoint 1 (run this instead of cells 2-6 to resume)
# ============================================================
CHECKPOINT_DIR = 'checkpoints'

item_embeddings = torch.load(f'{CHECKPOINT_DIR}/item_embeddings.pt', weights_only=True)
with open(f'{CHECKPOINT_DIR}/data_checkpoint1.pkl', 'rb') as f:
    _ckpt = pickle.load(f)
    games_df = _ckpt['games_df']
    user_sequences = _ckpt['user_sequences']
    sasrec_game_ids = _ckpt['sasrec_game_ids']
    del _ckpt

print(f'Checkpoint 1 loaded: item_embeddings {item_embeddings.shape}, '
      f'{len(games_df)} games, {len(user_sequences)} users')

Checkpoint 1 loaded: item_embeddings torch.Size([5232, 1024]), 5232 games, 56808 users


In [3]:
# ============================================================
# Load Checkpoint 2 (run this + checkpoint 1 load to skip RQVAE)
# ============================================================
CHECKPOINT_DIR = 'checkpoints'

with open(f'{CHECKPOINT_DIR}/data_checkpoint2.pkl', 'rb') as f:
    _ckpt = pickle.load(f)
    semantic_ids_3 = _ckpt['semantic_ids_3']
    semantic_ids_4 = _ckpt['semantic_ids_4']
    semantic_token_ids = _ckpt['semantic_token_ids']
    app_id_to_tokens = _ckpt['app_id_to_tokens']
    CODEBOOK_SIZE = _ckpt['CODEBOOK_SIZE']
    NUM_LEVELS = _ckpt['NUM_LEVELS']
    del _ckpt

print(f'Checkpoint 2 loaded: {len(app_id_to_tokens)} items, '
      f'{NUM_LEVELS} levels x {CODEBOOK_SIZE} codes')

Checkpoint 2 loaded: 5232 items, 4 levels x 256 codes


In [4]:
# ============================================================
# Load Checkpoint 3 (run this + checkpoints 1-2 to skip to training)
# ============================================================
CHECKPOINT_DIR = 'checkpoints'

with open(f'{CHECKPOINT_DIR}/data_checkpoint3.pkl', 'rb') as f:
    _ckpt = pickle.load(f)
    user_seqs = _ckpt['user_seqs']
    item_to_id = _ckpt['item_to_id']
    id_to_item = _ckpt['id_to_item']
    n_items = _ckpt['n_items']
    del _ckpt

MAX_SEQ_LEN = 100
print(f'Checkpoint 3 loaded: {len(user_seqs)} users, {n_items} items')

Checkpoint 3 loaded: 56692 users, 5232 items


In [5]:
# ============================================================
# Prepare semantic token sequences for TIGER
# ============================================================

# Convert user sequences to token sequences
user_token_seqs = {}
for user, seq in user_seqs.items():
    token_seq = []
    valid = True
    for item_id in seq:
        app_id = id_to_item[item_id]
        if app_id not in app_id_to_tokens:
            valid = False
            break
        token_seq.extend(app_id_to_tokens[app_id])
    if valid and len(seq) >= 3:
        user_token_seqs[user] = (token_seq, len(seq))

print(f'Users with semantic sequences: {len(user_token_seqs):,}')

# Build candidate pool
candidate_pool = list(set(tuple(v) for v in app_id_to_tokens.values()))
candidate_to_idx = {c: i for i, c in enumerate(candidate_pool)}
print(f'Candidate pool: {len(candidate_pool):,} unique items')


Users with semantic sequences: 56,692
Candidate pool: 5,232 unique items


---
## 8. TIGER — Generative Retrieval

TIGER flattens user history into a single token stream and does **next-token prediction** — exactly like a language model. At inference, it generates the next item's Semantic ID using **constrained beam search** over a trie of all valid Semantic IDs.

Key differences from Semantic SASRec:
- Input is a **flat token stream** (all levels concatenated), not item-level sequences
- Training objective is **cross-entropy on every token position**
- Inference uses **beam search** constrained by a trie, not candidate scoring

In [6]:
# ============================================================
# Semantic ID Trie for constrained beam search
# ============================================================

class SemanticIDTrie:
    """Prefix tree of all valid Semantic IDs for constrained generation."""
    
    def __init__(self, all_token_ids, num_levels):
        self.root = {}
        self.num_levels = num_levels
        self.token_to_item = {}
        
        for item_tokens in all_token_ids:
            node = self.root
            for level, token in enumerate(item_tokens):
                if token not in node:
                    node[token] = {}
                node = node[token]
            self.token_to_item[tuple(item_tokens)] = True
    
    def get_valid_tokens(self, prefix):
        """Return valid next tokens given a partial Semantic ID prefix."""
        node = self.root
        for token in prefix:
            if token not in node:
                return []
            node = node[token]
        return list(node.keys())
    
    def is_complete(self, tokens):
        return tuple(tokens) in self.token_to_item


# ============================================================
# TIGER Model — Generative Retrieval
# ============================================================

class TIGER(nn.Module):
    """TIGER: generative recommender with flat token stream and next-token prediction."""
    
    def __init__(self, vocab_size=1024, num_levels=4, codebook_size=256,
                 max_seq_length=100, hidden_dim=256, num_heads=4,
                 num_blocks=4, mlp_dim=512, dropout_rate=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.num_levels = num_levels
        self.codebook_size = codebook_size
        self.hidden_dim = hidden_dim
        self.max_token_len = max_seq_length * num_levels
        
        self.token_emb = nn.Embedding(vocab_size + 1, hidden_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(self.max_token_len + 1, hidden_dim, padding_idx=0)
        self.emb_dropout = nn.Dropout(dropout_rate)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=mlp_dim, dropout=dropout_rate,
            activation='relu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_blocks)
        self.ln_f = nn.LayerNorm(hidden_dim, eps=1e-8)
        self.lm_head = nn.Linear(hidden_dim, vocab_size, bias=False)
        
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)
            if m.padding_idx is not None:
                with torch.no_grad(): m.weight[m.padding_idx].fill_(0)
    
    def forward(self, input_ids):
        B, T = input_ids.size()
        embs = self.token_emb(input_ids) * self.hidden_dim**0.5
        positions = torch.arange(1, T + 1, device=input_ids.device).unsqueeze(0).expand(B, -1)
        positions = positions * (input_ids != 0).long()
        positions = positions.clamp(max=self.max_token_len)
        hidden = self.emb_dropout(embs + self.pos_emb(positions))
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=input_ids.device)
        hidden = self.transformer(hidden, mask=causal_mask, is_causal=True)
        hidden = self.ln_f(hidden)
        logits = self.lm_head(hidden)
        return logits
    
    def training_step(self, input_ids, target_ids, mask):
        """Next-token prediction loss on every position."""
        logits = self.forward(input_ids)
        loss = F.cross_entropy(
            logits.view(-1, self.vocab_size),
            target_ids.view(-1),
            reduction='none'
        )
        loss = (loss * mask.view(-1)).sum() / mask.sum().clamp(min=1)
        return loss
    
    @torch.no_grad()
    def beam_search(self, input_ids, trie, beam_width=10, num_levels=4):
        """Constrained beam search over trie of valid Semantic IDs."""
        logits = self.forward(input_ids)
        last_logits = logits[:, -1, :]
        log_probs = F.log_softmax(last_logits, dim=-1)
        
        B = input_ids.size(0)
        all_results = []
        
        for b in range(B):
            valid_l0 = trie.get_valid_tokens([])
            if not valid_l0:
                all_results.append([])
                continue
            
            valid_t = torch.tensor(valid_l0, device=input_ids.device)
            scores = log_probs[b, valid_t]
            topk = min(beam_width, len(valid_l0))
            top_scores, top_idx = scores.topk(topk)
            
            beams = [(top_scores[i].item(), [valid_l0[top_idx[i].item()]]) for i in range(topk)]
            
            for level in range(1, num_levels):
                new_beams = []
                for score, prefix in beams:
                    valid_next = trie.get_valid_tokens(prefix)
                    if not valid_next:
                        continue
                    
                    # Truncate history so total context fits within positional embedding range
                    max_hist = self.max_token_len - len(prefix)
                    hist = input_ids[b:b+1, -max_hist:]
                    ctx = torch.cat([
                        hist,
                        torch.tensor([prefix], device=input_ids.device)
                    ], dim=1)
                    
                    step_logits = self.forward(ctx)
                    step_lp = F.log_softmax(step_logits[0, -1, :], dim=-1)
                    
                    for tok in valid_next:
                        if tok >= self.vocab_size:
                            continue
                        new_score = score + step_lp[tok].item()
                        new_beams.append((new_score, prefix + [tok]))
                
                new_beams.sort(key=lambda x: x[0], reverse=True)
                beams = new_beams[:beam_width]
            
            results = [(score, tokens) for score, tokens in beams if trie.is_complete(tokens)]
            all_results.append(results)
        
        return all_results


print('TIGER model and SemanticIDTrie defined')

TIGER model and SemanticIDTrie defined


In [7]:
# ============================================================
# Build trie and prepare TIGER training data
# ============================================================

# Build trie from all valid Semantic IDs
all_sid_tuples = [tuple(v) for v in app_id_to_tokens.values()]
sid_trie = SemanticIDTrie(all_sid_tuples, NUM_LEVELS)
print(f'Trie built with {len(all_sid_tuples):,} items, {NUM_LEVELS} levels')

# Reverse lookup: token tuple -> app_id
tokens_to_app_id = {tuple(v): k for k, v in app_id_to_tokens.items()}


def make_tiger_batch(user_token_seqs, users, num_levels=4, max_seq_len=100):
    """Create TIGER training batch: flat token stream with next-token targets."""
    max_token_len = max_seq_len * num_levels
    B = len(users)
    
    input_ids = torch.zeros(B, max_token_len, dtype=torch.long)
    target_ids = torch.zeros(B, max_token_len, dtype=torch.long)
    mask = torch.zeros(B, max_token_len, dtype=torch.float)
    
    for i, user in enumerate(users):
        full_tokens, seq_len = user_token_seqs[user]
        train_len = min(seq_len - 1, max_seq_len)  # leave last item for target
        train_tokens = full_tokens[:train_len * num_levels]
        
        # Next item's tokens as final targets
        next_start = train_len * num_levels
        next_tokens = full_tokens[next_start:next_start + num_levels]
        
        # Input: history tokens + next item tokens (except last)
        all_input = train_tokens + next_tokens[:-1]
        all_target = train_tokens[1:] + next_tokens
        
        tl = min(len(all_input), max_token_len)
        input_ids[i, -tl:] = torch.tensor(all_input[-tl:])
        target_ids[i, -tl:] = torch.tensor(all_target[-tl:])
        mask[i, -tl:] = 1.0
    
    return input_ids, target_ids, mask


def evaluate_tiger(model, user_token_seqs, trie, tokens_to_app_id, 
                   mode='val', num_levels=4, max_seq_len=100, 
                   beam_width=10, device='cpu'):
    """Evaluate TIGER with constrained beam search."""
    model.eval()
    ranks = []
    users = list(user_token_seqs.keys())
    
    for user in tqdm(users, desc=f'TIGER eval ({mode})', leave=False):
        full_tokens, seq_len = user_token_seqs[user]
        
        if mode == 'val':
            if seq_len < 3: continue
            split = seq_len - 2
        else:
            if seq_len < 2: continue
            split = seq_len - 1
        
        inp = full_tokens[:split * num_levels]
        tgt = tuple(full_tokens[split * num_levels:(split + 1) * num_levels])
        
        max_token_len = max_seq_len * num_levels
        tokens = inp[-max_token_len:] if len(inp) > max_token_len else inp
        input_t = torch.zeros(1, len(tokens), dtype=torch.long, device=device)
        input_t[0] = torch.tensor(tokens, device=device)
        
        results = model.beam_search(input_t, trie, beam_width=beam_width, num_levels=num_levels)
        
        if not results[0]:
            ranks.append(beam_width + 1)
            continue
        
        generated = [tuple(tokens) for _, tokens in results[0]]
        if tgt in generated:
            rank = generated.index(tgt) + 1
        else:
            rank = beam_width + 1
        ranks.append(rank)
    
    ranks = np.array(ranks)
    return {
        'hr@10': (ranks <= 10).mean(),
        'ndcg@10': np.where(ranks <= 10, 1.0 / np.log2(ranks + 1), 0.0).mean()
    }


print('TIGER training and evaluation functions defined')

Trie built with 5,232 items, 4 levels
TIGER training and evaluation functions defined


In [ ]:
# ============================================================
# Train TIGER
# ============================================================

tiger = TIGER(
    vocab_size=NUM_LEVELS * CODEBOOK_SIZE,
    num_levels=NUM_LEVELS,
    codebook_size=CODEBOOK_SIZE,
    max_seq_length=MAX_SEQ_LEN,
    hidden_dim=256, num_heads=4,
    num_blocks=2, mlp_dim=512, dropout_rate=0.1
).to(device)

print(f'TIGER params: {sum(p.numel() for p in tiger.parameters()):,}')

opt_tiger = torch.optim.AdamW(tiger.parameters(), lr=1e-3, betas=(0.9, 0.98))
all_tiger_users = list(user_token_seqs.keys())
BATCH_SIZE_TIGER = 128
NUM_EPOCHS_TIGER = 200

best_tiger_ndcg = 0
for epoch in tqdm(range(NUM_EPOCHS_TIGER), desc="TIGER Training"):
    tiger.train()
    np.random.shuffle(all_tiger_users)
    epoch_loss = 0
    n_batches = 0
    
    for i in range(0, len(all_tiger_users), BATCH_SIZE_TIGER):
        batch_users = all_tiger_users[i:i + BATCH_SIZE_TIGER]
        inp, tgt, mask = make_tiger_batch(user_token_seqs, batch_users, NUM_LEVELS, MAX_SEQ_LEN)
        inp, tgt, mask = inp.to(device), tgt.to(device), mask.to(device)
        
        opt_tiger.zero_grad()
        loss = tiger.training_step(inp, tgt, mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tiger.parameters(), 1.0)
        opt_tiger.step()
        epoch_loss += loss.item()
        n_batches += 1
    
    if (epoch + 1) % 20 == 0:
        val = evaluate_tiger(tiger, user_token_seqs, sid_trie, tokens_to_app_id,
                             'val', NUM_LEVELS, MAX_SEQ_LEN, beam_width=10, device=device)
        print(f'Epoch {epoch+1:3d} | loss: {epoch_loss/max(n_batches,1):.4f} | '
              f'val HR@10: {val["hr@10"]:.4f} | NDCG@10: {val["ndcg@10"]:.4f}')
        if val['ndcg@10'] > best_tiger_ndcg:
            best_tiger_ndcg = val['ndcg@10']
            torch.save(tiger.state_dict(), 'tiger_best.pt')

# Final test
tiger.load_state_dict(torch.load('tiger_best.pt', map_location=device, weights_only=True))
test_tiger = evaluate_tiger(tiger, user_token_seqs, sid_trie, tokens_to_app_id,
                            'test', NUM_LEVELS, MAX_SEQ_LEN, beam_width=10, device=device)
print(f'\nTIGER Test: HR@10={test_tiger["hr@10"]:.4f}, NDCG@10={test_tiger["ndcg@10"]:.4f}')

TIGER params: 1,681,920


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


TIGER Training:   0%|          | 0/200 [00:00<?, ?it/s]

TIGER eval (val):   0%|          | 0/56692 [00:00<?, ?it/s]

Epoch  20 | loss: 1.4015 | val HR@10: 0.1574 | NDCG@10: 0.0835


...

Epoch 180 | loss: 1.3413 | val HR@10: 0.1356 | NDCG@10: 0.0697


TIGER eval (val):   0%|          | 0/56692 [00:00<?, ?it/s]

Epoch 200 | loss: 1.3392 | val HR@10: 0.1288 | NDCG@10: 0.0666


TIGER eval (test):   0%|          | 0/56692 [00:00<?, ?it/s]


TIGER Test: HR@10=0.1447, NDCG@10=0.0764
